# Model Performance Benchmarking

**The ROI of Quantization** — Run and analyze GuideLLM benchmarks to measure vLLM inference performance on RHOAI 3.3.

## What This Notebook Does

1. **Check Baseline** — Verify which models are running and their GPU allocation
2. **Run Benchmarks** — Execute GuideLLM tests against deployed models
3. **Analyze Results** — Parse benchmark JSON from CronJob results
4. **Visualize Metrics** — Plot TTFT, TPOT, throughput curves
5. **Compare Models** — INT4 vs BF16 efficiency and ROI analysis

---

### Volume Mounts

| Path | Source | Contents |
|------|--------|----------|
| `/results` | `guidellm-results` PVC | CronJob and Job template benchmark results |
| `/opt/app-root/src` | Workbench storage | Notebooks and analysis |

### GPU Configuration

| Node | GPUs | Active Model | Role |
|------|------|-------------|------|
| g6.4xlarge | 1x L4 | `qwen3-8b-agent` | RAG, MCP, Guardrails |
| g6.12xlarge | 4x L4 | `mistral-3-bf16` | Full-precision LLM |
| **Total** | **5 GPUs** | | 64 vCPU sandbox limit |

---
## 1. Setup Environment

In [ ]:
!pip install -q pandas matplotlib plotly requests tabulate

In [ ]:
import os
import json
import glob
import subprocess
from datetime import datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("/results")

MODELS = {
    "qwen3-8b-agent": {
        "url": "http://qwen3-8b-agent-predictor.private-ai.svc.cluster.local:8080/v1",
        "gpus": 1, "node": "g6.4xlarge", "baseline": True,
    },
    "mistral-3-bf16": {
        "url": "http://mistral-3-bf16-predictor.private-ai.svc.cluster.local:8080/v1",
        "gpus": 4, "node": "g6.12xlarge", "baseline": True,
    },
    "mistral-3-int4": {
        "url": "http://mistral-3-int4-predictor.private-ai.svc.cluster.local:8080/v1",
        "gpus": 1, "node": "g6.4xlarge", "baseline": False,
    },
    "devstral-2": {
        "url": "http://devstral-2-predictor.private-ai.svc.cluster.local:8080/v1",
        "gpus": 4, "node": "g6.12xlarge", "baseline": False,
    },
    "gpt-oss-20b": {
        "url": "http://gpt-oss-20b-predictor.private-ai.svc.cluster.local:8080/v1",
        "gpus": 4, "node": "g6.12xlarge", "baseline": False,
    },
}

print(f"Results dir:       {RESULTS_DIR}")
print(f"Models configured: {len(MODELS)}")

---
## 2. Check Model Status

In [ ]:
!oc get inferenceservice -n private-ai -o custom-columns='NAME:.metadata.name,READY:.status.conditions[?(@.type=="Ready")].status,MIN:.spec.predictor.minReplicas'

In [ ]:
import requests

print("Model Health Check")
print("=" * 60)
for name, cfg in MODELS.items():
    tag = "[ACTIVE]" if cfg["baseline"] else "[QUEUED]"
    try:
        resp = requests.get(f"{cfg['url']}/models", timeout=3)
        if resp.status_code == 200:
            models = resp.json().get("data", [])
            print(f"  PASS  {name:<20} {tag}  {len(models)} model(s) loaded")
        else:
            print(f"  WARN  {name:<20} {tag}  HTTP {resp.status_code}")
    except Exception:
        print(f"  ---   {name:<20} {tag}  not running")

---
## 3. Run Benchmark

### Option A: Trigger CronJob (parallel, all active models)

In [ ]:
!oc create job --from=cronjob/guidellm-daily benchmark-nb-$(date +%H%M%S) -n private-ai
print("\nMonitor: oc logs -f job/benchmark-nb-... -n private-ai")

### Option B: Job Template (single model, on-demand)

In [ ]:
# Choose the model to benchmark:
#   qwen3-8b-agent.yaml  |  mistral-3-bf16.yaml

TARGET = "qwen3-8b-agent.yaml"
!oc create -f gitops/step-06-model-metrics/base/guidellm/job-templates/{TARGET}
!oc get jobs -n private-ai -l app=guidellm --sort-by=.metadata.creationTimestamp | tail -5

---
## 4. Analyze Benchmark Results

In [ ]:
def list_results(results_dir: Path, limit: int = 10) -> list:
    if not results_dir.exists():
        return []
    files = sorted(results_dir.glob("**/*.json"), key=os.path.getmtime, reverse=True)
    return files[:limit]

print("Benchmark Results (most recent):")
for f in list_results(RESULTS_DIR, 10):
    size = f.stat().st_size / 1024
    mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
    print(f"  {f.name:<40} {size:>8.1f} KB  {mtime}")

In [ ]:
def parse_guidellm_results(filepath: Path) -> dict:
    """Parse GuideLLM benchmark results JSON."""
    with open(filepath) as f:
        data = json.load(f)

    benchmarks = data.get("benchmarks", [])
    results = []

    for bench in benchmarks:
        metrics = bench.get("metrics", {})
        results.append({
            "rate": bench.get("rate", 0),
            "rate_type": bench.get("rate_type", "unknown"),
            "completed_requests": metrics.get("request_count", 0),
            "throughput_tok_s": metrics.get("output_token_throughput", {}).get("mean", 0),
            "ttft_p50_ms": metrics.get("ttft", {}).get("p50", 0) * 1000,
            "ttft_p95_ms": metrics.get("ttft", {}).get("p95", 0) * 1000,
            "tpot_p50_ms": metrics.get("itl", {}).get("p50", 0) * 1000,
            "tpot_p95_ms": metrics.get("itl", {}).get("p95", 0) * 1000,
        })

    return {
        "model": data.get("model", "unknown"),
        "target": data.get("target", "unknown"),
        "benchmarks": results
    }

recent_files = list_results(CRONJOB_RESULTS)
if recent_files:
    result = parse_guidellm_results(recent_files[0])
    print(f"Latest: {recent_files[0].name}")
    print(f"Model:  {result['model']}")
    print(f"Rates:  {len(result['benchmarks'])} levels")
    df = pd.DataFrame(result["benchmarks"])
    print("\n" + df.to_string(index=False))
else:
    print("No results found. Run a benchmark first (Section 3).")

---
## 5. Visualize Performance Metrics

In [ ]:
def plot_latency_vs_concurrency(results: list, title: str = "Latency vs Concurrency"):
    df = pd.DataFrame(results)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(df["rate"], df["ttft_p50_ms"], "b-o", label="P50")
    axes[0].plot(df["rate"], df["ttft_p95_ms"], "r-s", label="P95")
    axes[0].axhline(y=1000, color="orange", linestyle="--", label="SLA (1s)")
    axes[0].axhline(y=2000, color="red", linestyle="--", label="Breaking (2s)")
    axes[0].set_xlabel("Concurrent Users")
    axes[0].set_ylabel("TTFT (ms)")
    axes[0].set_title("Time to First Token")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(df["rate"], df["tpot_p50_ms"], "b-o", label="P50")
    axes[1].plot(df["rate"], df["tpot_p95_ms"], "r-s", label="P95")
    axes[1].axhline(y=100, color="orange", linestyle="--", label="SLA (100ms)")
    axes[1].set_xlabel("Concurrent Users")
    axes[1].set_ylabel("TPOT (ms)")
    axes[1].set_title("Time per Output Token")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

if recent_files:
    result = parse_guidellm_results(recent_files[0])
    if result["benchmarks"]:
        plot_latency_vs_concurrency(result["benchmarks"], f"Model: {result['model']}")

In [ ]:
def plot_throughput_curve(results: list, title: str = "Throughput vs Concurrency"):
    df = pd.DataFrame(results)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df["rate"], df["throughput_tok_s"], "g-o", linewidth=2, markersize=8)
    ax.fill_between(df["rate"], df["throughput_tok_s"], alpha=0.3, color="green")

    ax.set_xlabel("Concurrent Users", fontsize=12)
    ax.set_ylabel("Throughput (tokens/sec)", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.grid(True, alpha=0.3)

    max_idx = df["throughput_tok_s"].idxmax()
    max_rate = df.loc[max_idx, "rate"]
    max_tp = df.loc[max_idx, "throughput_tok_s"]
    ax.annotate(f"Peak: {max_tp:.0f} tok/s @ {max_rate} users",
                xy=(max_rate, max_tp),
                xytext=(max_rate + 2, max_tp * 0.9),
                fontsize=10,
                arrowprops=dict(arrowstyle="->", color="black"))

    plt.tight_layout()
    plt.show()

if recent_files:
    result = parse_guidellm_results(recent_files[0])
    if result["benchmarks"]:
        plot_throughput_curve(result["benchmarks"], f"Throughput: {result['model']}")

---
## 6. ROI Analysis: INT4 vs BF16

Calculate the economics of quantization — can 4x INT4 instances outperform 1x BF16 at the same cost?

In [ ]:
COSTS = {
    "int4": {"gpus": 1, "instance": "g6.4xlarge", "cost_hr": 0.85},
    "bf16": {"gpus": 4, "instance": "g6.12xlarge", "cost_hr": 3.40},
}

def find_model_results(model_substr: str, results_dir: Path) -> Path:
    files = sorted(results_dir.glob(f"*{model_substr}*.json"), key=os.path.getmtime, reverse=True)
    return files[0] if files else None

int4_file = find_model_results("int4", CRONJOB_RESULTS)
bf16_file = find_model_results("bf16", CRONJOB_RESULTS)

print("Result files:")
print(f"  INT4: {int4_file.name if int4_file else 'Not found'}")
print(f"  BF16: {bf16_file.name if bf16_file else 'Not found'}")

In [ ]:
def calculate_roi(int4_results: dict, bf16_results: dict):
    int4_df = pd.DataFrame(int4_results["benchmarks"])
    bf16_df = pd.DataFrame(bf16_results["benchmarks"])

    int4_peak = int4_df["throughput_tok_s"].max()
    bf16_peak = bf16_df["throughput_tok_s"].max()

    int4_eff = int4_peak / COSTS["int4"]["cost_hr"]
    bf16_eff = bf16_peak / COSTS["bf16"]["cost_hr"]

    print("\nROI Analysis: The Economics of Precision")
    print("=" * 55)
    print(f"\n{'Metric':<30} {'INT4 (1-GPU)':<15} {'BF16 (4-GPU)':<15}")
    print("-" * 60)
    print(f"{'Hardware Cost':<30} ${COSTS['int4']['cost_hr']:.2f}/hr{'':<9} ${COSTS['bf16']['cost_hr']:.2f}/hr")
    print(f"{'Peak Throughput':<30} {int4_peak:.0f} tok/s{'':<7} {bf16_peak:.0f} tok/s")
    print(f"{'Efficiency (tok/s per $)':<30} {int4_eff:.0f}{'':<14} {bf16_eff:.0f}")
    print(f"{'Cost Ratio':<30} 1x{'':<15} {COSTS['bf16']['cost_hr']/COSTS['int4']['cost_hr']:.1f}x")

    four_int4_tp = int4_peak * 4
    print("\n" + "=" * 55)
    print("\nKey Insight: 4x INT4 vs 1x BF16 (Same Cost)")
    print(f"  4x INT4 Throughput: {four_int4_tp:.0f} tok/s")
    print(f"  1x BF16 Throughput: {bf16_peak:.0f} tok/s")
    if four_int4_tp > bf16_peak:
        print(f"  Advantage: INT4 delivers {(four_int4_tp/bf16_peak - 1)*100:.0f}% more throughput")
    else:
        print(f"  Note: BF16 delivers {(bf16_peak/four_int4_tp - 1)*100:.0f}% more throughput")

if int4_file and bf16_file:
    calculate_roi(parse_guidellm_results(int4_file), parse_guidellm_results(bf16_file))
else:
    print("Need both INT4 and BF16 results for ROI analysis.")
    print("Run benchmarks for both models first (Section 3).")

---
## References

- [RHOAI 3.3: Monitoring Models](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.3/html/managing_and_monitoring_models/)
- [GuideLLM Documentation](https://github.com/neuralmagic/guidellm)
- [vLLM Production Metrics](https://docs.vllm.ai/en/latest/serving/metrics.html)
- [GuideLLM - Evaluate LLM Deployments (Red Hat Developers)](https://developers.redhat.com/articles/2025/06/20/guidellm-evaluate-llm-deployments-real-world-inference)